# 🏨 Agoda Review Crawler (hotel)

Chạy lần lượt các cell từ trên xuống: **① Cấu hình → ② Đọc input → ③ Crawl → ④ Xem kết quả**.

**Input:** Google Sheet (cột `Hotel` + `URL` Agoda) hoặc file CSV/XLSX/TXT offline (sửa `INPUT_MODE` ở cell ①). Sheet mẫu: [agoda-review](https://docs.google.com/spreadsheets/d/1Di_rPSgFJA-UGaDElWTLnn_A10yfQRIOVeu65HjCUJA/edit?gid=1289817800#gid=1289817800).

**Không cần browser:** crawl trực tiếp API Agoda (`HotelReviews` + `ReviewComments`) bằng curl_cffi — cùng kỹ thuật capture & replay như crawler giá / Google Maps review.

**Output** nằm trong `results/agoda-review/<RUN_NAME>/`:
- `FINAL_hotels_<YYYYMMDD>.csv` — 1 dòng/hotel: rating tổng, số đánh giá, sub-ratings…
- `FINAL_reviews_<YYYYMMDD>.csv` — 1 dòng/review: nguồn, rating, title, text, reviewer, room, reply…
- `TEMP_agoda_*.csv` — checkpoint: lỡ tắt giữa chừng, chạy lại cell ③ sẽ tự resume — hotel **chưa từng cào** chạy trước, hotel lỗi retry sau.

ℹ️ Mặc định chỉ lấy review **Agoda** (`provider=agoda`). Đặt `PROVIDER = "all"` để lấy thêm Booking.com / Priceline… `review_count` là tổng điểm trên placecard; `fetched_reviews` là số comment thực lấy được từ provider đã chọn.

In [1]:
# ════════════════ ① CẤU HÌNH ════════════════

# ── Tên run: kết quả nằm RIÊNG trong results/agoda-review/<RUN_NAME>/ ──
RUN_NAME = "run1"

# ── Nguồn input: "gsheet" (online) hoặc "offline" (file trên máy) ──
INPUT_MODE = "gsheet"

# Dùng khi INPUT_MODE = "gsheet" (gid của tab được tự lấy từ URL)
GSHEET_URL = "https://docs.google.com/spreadsheets/d/1Di_rPSgFJA-UGaDElWTLnn_A10yfQRIOVeu65HjCUJA/edit?gid=1289817800#gid=1289817800"

# Dùng khi INPUT_MODE = "offline" — đường dẫn tuyệt đối, hoặc tương đối so với 31.crawl-tool
# File cần có cột URL Agoda (cột Hotel/name nếu có sẽ được giữ)
OFFLINE_FILE = "input/agoda_review_hotels.csv"

# ── Tham số crawl ──
MAX_REVIEWS = 0          # số review tối đa mỗi hotel; 0 = lấy HẾT (theo provider)
MAX_HOTELS  = 0          # 0 = crawl tất cả; đặt 1–2 để test nhanh
SORT        = "recent"   # recent | helpful | rating_high | rating_low
PROVIDER    = "agoda"    # "agoda" | "all" | hoặc providerId số (332=Agoda, 3038=Booking)

In [2]:
# ════════════════ ② ĐỌC INPUT ════════════════
import importlib
import os, sys

if "ROOT" not in globals():                    # giữ nguyên ROOT khi chạy lại cell
    NB_DIR = os.path.abspath("")
    ROOT = NB_DIR if os.path.isdir(os.path.join(NB_DIR, "crawler")) else os.path.dirname(NB_DIR)
assert os.path.isdir(os.path.join(ROOT, "crawler")), (
    f"Không tìm thấy package `crawler` quanh {ROOT} — hãy mở notebook từ 31.crawl-tool/agoda-review")
for p in (ROOT, os.path.join(ROOT, "agoda-review")):
    if p not in sys.path:
        sys.path.insert(0, p)

import agoda_review
importlib.reload(agoda_review)                 # nhận thay đổi nếu vừa sửa agoda_review.py

if INPUT_MODE == "gsheet":
    INPUT = GSHEET_URL
    print("📡 Input: Google Sheet online")
else:
    INPUT = OFFLINE_FILE if os.path.isabs(OFFLINE_FILE) else os.path.join(ROOT, OFFLINE_FILE)
    assert os.path.exists(INPUT), f"Không tìm thấy file: {INPUT}"
    print(f"📁 Input: file offline — {INPUT}")

links = agoda_review.read_links(INPUT)
print(f"✅ Đọc được {len(links)} hotel (đã dedupe URL). 5 dòng đầu:")
for url, name in links[:5]:
    print(f"   • {name or '(tên tự lấy từ Agoda)'} — {url[:90]}")

📡 Input: Google Sheet online
✅ Đọc được 3 hotel (đã dedupe URL). 5 dòng đầu:
   • Almanty - Junior Suite — https://www.agoda.com/vi-vn/almanity-hoi-an-wellness-resort-spa-inclusive/hotel/hoi-an-vn.
   • Au Lac Charner — https://www.agoda.com/en-gb/silverland-charner/hotel/ho-chi-minh-city-vn.html?finalPriceVi
   • Wink — https://www.agoda.com/en-gb/wink-hotel-saigon-centre/hotel/ho-chi-minh-city-vn.html?finalP


In [3]:
# (TÙY CHỌN) Tải Google Sheet về file offline — lần sau chỉ cần đổi INPUT_MODE = "offline"
import pandas as pd
from crawler.hotels_io import _gsheet_url

os.makedirs(os.path.join(ROOT, "input"), exist_ok=True)
dest = os.path.join(ROOT, "input", "agoda_review_hotels.csv")
pd.read_csv(_gsheet_url(GSHEET_URL)).to_csv(dest, index=False, encoding="utf-8-sig")
print(f"💾 Đã lưu bản offline: {dest}")

💾 Đã lưu bản offline: /Users/hchinhtrung/Documents/GitHub/mvillage-email-template/31.crawl-tool/input/agoda_review_hotels.csv


In [4]:
# ════════════════ ③ CRAWL ════════════════
OUTDIR = os.path.join(ROOT, "results", "agoda-review", RUN_NAME)

agoda_review.crawl(
    INPUT,
    out_dir=OUTDIR,
    max_reviews=MAX_REVIEWS,
    max_hotels=MAX_HOTELS,
    sort=SORT,
    provider=PROVIDER,
)

🚀 3 hotel trong input | crawl 3 (mới 3, retry 0) | max_reviews=ALL | sort=recent(1) | provider=agoda

🏨 1/3 Almanty - Junior Suite
   ⭐ 9.0 (Exceptional) | 6206 đánh giá | propertyId=686575
   📊 Cleanliness:9.2;Facilities:9.1;Location:9.1;Room comfort and quality:8.9;Service:9.3;Value
   … đã lấy 200 review
   … đã lấy 400 review
   … đã lấy 600 review
   … đã lấy 800 review
   … đã lấy 1000 review
   … đã lấy 1200 review
   … đã lấy 1400 review
   … đã lấy 1600 review
   ✅ lấy được 3002 review

🏨 2/3 Au Lac Charner
   ⭐ 8.9 (Excellent) | 8552 đánh giá | propertyId=5794766
   📊 Cleanliness:8.9;Facilities:8.6;Location:9.4;Service:9.2;Value for money:8.7
   … đã lấy 200 review
   … đã lấy 400 review
   … đã lấy 600 review
   … đã lấy 800 review
   … đã lấy 1000 review
   … đã lấy 1200 review
   … đã lấy 1400 review
   … đã lấy 1600 review
   … đã lấy 1800 review
   … đã lấy 2000 review
   … đã lấy 2200 review
   … đã lấy 2400 review
   … đã lấy 2600 review
   … đã lấy 2800 review
   … đã

('/Users/hchinhtrung/Documents/GitHub/mvillage-email-template/31.crawl-tool/results/agoda-review/run1/FINAL_hotels_20260716.csv',
 '/Users/hchinhtrung/Documents/GitHub/mvillage-email-template/31.crawl-tool/results/agoda-review/run1/FINAL_reviews_20260716.csv')

In [5]:
# ════════════════ ④ XEM KẾT QUẢ ════════════════
import glob
import pandas as pd
from IPython.display import display

hotels_files = sorted(glob.glob(os.path.join(OUTDIR, "FINAL_hotels_*.csv")))
reviews_files = sorted(glob.glob(os.path.join(OUTDIR, "FINAL_reviews_*.csv")))
assert hotels_files and reviews_files, f"Chưa có FINAL_* trong {OUTDIR} — chạy cell ③ trước"

hf, rf = hotels_files[-1], reviews_files[-1]
hotels = pd.read_csv(hf)
reviews = pd.read_csv(rf)
src = reviews["source"].fillna("(blank)").value_counts().to_dict() if len(reviews) else {}
print(f"📄 {hf} — {len(hotels)} hotel")
print(f"📄 {rf} — {len(reviews)} review", end="")
if src:
    print(" (" + ", ".join(f"{k}: {v}" for k, v in src.items()) + ")")
else:
    print()

display(hotels[[c for c in ["hotel_name", "overall_rating", "rating_text",
                            "review_count", "fetched_reviews", "status", "subratings"]
                if c in hotels.columns]].head(20))
display(reviews[[c for c in ["hotel_name", "source", "author", "rating", "title",
                             "review_date", "country", "traveler_type", "text"]
                 if c in reviews.columns]].head(10))

📄 /Users/hchinhtrung/Documents/GitHub/mvillage-email-template/31.crawl-tool/results/agoda-review/run1/FINAL_hotels_20260716.csv — 3 hotel
📄 /Users/hchinhtrung/Documents/GitHub/mvillage-email-template/31.crawl-tool/results/agoda-review/run1/FINAL_reviews_20260716.csv — 9311 review (Agoda: 6045, Booking.com: 3266)


,hotel_name,overall_rating,rating_text,review_count,fetched_reviews,status,subratings
0,Almanty - Junior Suite,9.0,Exceptional,6206,3002,ok,Cleanliness:9.2;Facilities:9.1;Location:9.1;Ro...
1,Au Lac Charner,8.9,Excellent,8552,4210,ok,Cleanliness:8.9;Facilities:8.6;Location:9.4;Se...
2,Wink,8.8,Excellent,7371,2099,ok,Cleanliness:9.3;Facilities:8.9;Location:8.8;Se...


,hotel_name,source,author,rating,title,review_date,country,traveler_type,text
0,Almanty - Junior Suite,Agoda,Ulrike,9.2,Relaxing stay,2026-07-07,Australia,Couple,We stayed at Almanity for 3 nights in the heat...
1,Almanty - Junior Suite,Booking.com,Kevin,9.0,An amazingly nice hotel…,2026-06-29,NaN,Couple,NaN
2,Almanty - Junior Suite,Agoda,Akarshan,10.0,Peaceful Stay,2026-07-04,India,Couple,"It’s very good hotel with great people. Stay, ..."
3,Almanty - Junior Suite,Booking.com,RICHARD,10.0,Very nice,2026-06-05,NaN,Solo traveler,NaN
4,Almanty - Junior Suite,Agoda,Gus,10.0,Lots of shade,2026-07-03,Australia,Family with teens,I really enjoyed the Pool area as it was in th...
5,Almanty - Junior Suite,Booking.com,Cameron,8.0,"Good place, but probably better value for money",2026-04-02,NaN,Couple,NaN
6,Almanty - Junior Suite,Agoda,Hinda,10.0,Great stay,2026-07-03,United States,Solo traveler,Great stay in a great location. Staff were gre...
7,Almanty - Junior Suite,Booking.com,Alisac,9.0,Excellent,2026-03-27,NaN,Couple,NaN
8,Almanty - Junior Suite,Agoda,Jimmy,10.0,Es recomendado. El valor es mas alto que el pr...,2026-06-28,Australia,Group,"Es un buen lugar para hospdarse , tiene restau..."
9,Almanty - Junior Suite,Booking.com,Sebastian,10.0,Amazing hotel experience in Hoi An,2026-03-23,NaN,Couple,NaN
